# Research 09 — LightGBM random-seed diversity

**Hypothesis:** changing LightGBM's random seed changes feature subsampling enough to yield complementary OOF errors, even with the selected structure and features held fixed. Train one 3-fold candidate with seed 2025 (baseline seed 42), then test a predeclared 80/20 blend with the current ensemble. This is a single seed-diversity test, not a seed sweep. Test is not loaded.

In [1]:
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
from IPython.display import display
from lightgbm import LGBMClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "data" / "train.csv").is_file()
)
train = pd.read_csv(ROOT / "data/train.csv")
y = train.segment.to_numpy(dtype=int)
base = pd.read_csv(ROOT / "predictions/segment-oof.csv")
assert len(base) == len(y) and np.array_equal(base.segment.to_numpy(), y)
p_base = base[[f"prob_{i}" for i in range(4)]].to_numpy(dtype=float)


def features(frame):
    x = frame.drop(
        columns=["segment", "player_id", "random_metric_1", "random_metric_2", "random_metric_3"],
        errors="ignore",
    ).copy()
    ratios = {
        "spend_per_hour": ("total_spending_thb", "total_playtime_hours"),
        "spend_per_account_day": ("total_spending_thb", "account_age_days"),
        "playtime_per_account_day": ("total_playtime_hours", "account_age_days"),
        "monthly_spend_per_purchase": ("avg_monthly_spending", "spending_frequency"),
        "gifts_per_friend": ("gifts_sent_received", "friend_count"),
        "invites_per_friend": ("friend_invites_sent", "friend_count"),
        "ranked_per_team": ("ranked_participation_rate", "team_play_percentage"),
    }
    for name, (numerator, denominator) in ratios.items():
        x[name] = x[numerator] / (x[denominator].abs() + 1)
    x["weekly_minutes"] = x["play_frequency"] * x["avg_session_duration"]
    x["social_activity"] = x["friend_count"] * x["team_play_percentage"]
    x["missing_count"] = x.isna().sum(axis=1)
    return x.replace([np.inf, -np.inf], np.nan)


X = features(train)
cat_cols = X.select_dtypes(include=["object", "string"]).columns.tolist()
print(f"Features={X.shape[1]}; categoricals={len(cat_cols)}; candidate seed=2025")

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
p_seed = np.zeros((len(y), 4), dtype=float)
fold_scores = []
started = perf_counter()
for fold, (fit_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
    x_fit, x_valid = X.iloc[fit_idx].copy(), X.iloc[valid_idx].copy()
    for col in cat_cols:
        categories = sorted(x_fit[col].dropna().unique().tolist())
        dtype = pd.CategoricalDtype(categories=categories)
        x_fit[col] = x_fit[col].astype(dtype)
        x_valid[col] = x_valid[col].astype(dtype)
    model = LGBMClassifier(
        objective="multiclass",
        n_estimators=440,
        learning_rate=0.063863707925569,
        num_leaves=59,
        max_depth=4,
        min_child_samples=81,
        reg_lambda=0.12942956115511214,
        colsample_bytree=0.7975990992289793,
        random_state=2025,
        n_jobs=2,
        verbosity=-1,
    )
    model.fit(x_fit, y[fit_idx], categorical_feature=cat_cols)
    p_seed[valid_idx] = model.predict_proba(x_valid)
    fold_score = f1_score(y[valid_idx], p_seed[valid_idx].argmax(axis=1), average="macro")
    fold_scores.append(fold_score)
    print(f"Fold {fold}: macro F1={fold_score:.6f}", flush=True)

p_blend = 0.8 * p_base + 0.2 * p_seed
rows = []
folds = list(cv.split(X, y))
for name, probs in [
    ("Current CatBoost/LightGBM", p_base),
    ("LightGBM seed 2025", p_seed),
    ("80/20 current + seed 2025", p_blend),
]:
    pred = probs.argmax(axis=1)
    row = {
        "candidate": name,
        "macro_f1": f1_score(y, pred, average="macro"),
        "weighted_f1": f1_score(y, pred, average="weighted"),
    }
    for fold, (_, valid_idx) in enumerate(folds, start=1):
        row[f"fold_{fold}_macro_f1"] = f1_score(y[valid_idx], pred[valid_idx], average="macro")
    rows.append(row)
results = pd.DataFrame(rows)
base_folds = results.loc[
    results.candidate == "Current CatBoost/LightGBM",
    [f"fold_{fold}_macro_f1" for fold in range(1, 4)],
].iloc[0]
for fold in range(1, 4):
    results[f"fold_{fold}_delta"] = (
        results[f"fold_{fold}_macro_f1"] - base_folds[f"fold_{fold}_macro_f1"]
    )
display(results.round(6))
results.to_csv(ROOT / "assets/segment-seed-diversity-results.csv", index=False)
for name, probs in [("LightGBM seed 2025", p_seed), ("80/20 current + seed 2025", p_blend)]:
    suffix = "seed-2025" if name.startswith("LightGBM") else "seed-2025-blend"
    pd.DataFrame(probs, columns=[f"prob_{i}" for i in range(4)]).assign(segment=y).to_csv(
        ROOT / f"predictions/segment-lightgbm-{suffix}-oof.csv", index=False
    )
print(f"Elapsed seconds={perf_counter() - started:.1f}; fold macro F1={fold_scores}")
print("OOF screening only. Test and submit/index.csv were not loaded or modified.")

Features=51; categoricals=10; candidate seed=2025


Fold 1: macro F1=0.744082


Fold 2: macro F1=0.748811


Fold 3: macro F1=0.749791


,candidate,macro_f1,weighted_f1,fold_1_macro_f1,fold_2_macro_f1,fold_3_macro_f1,fold_1_delta,fold_2_delta,fold_3_delta
0,Current CatBoost/LightGBM,0.750895,0.764683,0.749445,0.749821,0.753390,0.000000,0.000000,0.000000
1,LightGBM seed 2025,0.747572,0.761689,0.744082,0.748811,0.749791,-0.005363,-0.001010,-0.003599
2,80/20 current + seed 2025,0.750666,0.764497,0.749135,0.749946,0.752884,-0.000310,0.000125,-0.000506


Elapsed seconds=105.6; fold macro F1=[0.744081901580335, 0.7488106605413472, 0.749790719422546]
OOF screening only. Test and submit/index.csv were not loaded or modified.


## Decision

Compare the one alternate-seed model and its fixed 20% blend against the current OOF ensemble. Seed-level improvements are not enough on their own: they must also provide stable fold gains and be independently confirmed before model promotion.